<div dir="rtl">

# HM_03 — פתרון: מעבדה: האם פרומפט מובנה באמת טוב יותר? למדוד במקום לנחש

**הקורס:** מודלי שפה וסוכנים ממוחשבים למסלול מערכות מידע · **מפגש 3:** Prompt Engineering לקוד

**שם:** פתרון למרצה — לא להפצה לסטודנטים

---

## מה עושים במעבדה

בהרצאה הגדרנו את איכות הפרומפט $x$ כ**הסתברות ההצלחה** שלו, $P_\text{succ}(x)$: ההסתברות שהתשובה שהמודל ייצר תעמוד במפרט. במעבדה הזו תבנו את הכלים למדוד אותה, ותשתמשו בהם בניסוי אמיתי: הפרומפט הגרוע `תעשה בדיקת ראשוניות` (נסמן $x_A$) מול פרומפט מובנה שתכתבו (נסמן $x_B$).

| חלק | נושא | מה מממשים | זמן |
|---|---|---|---|
| א' | ארגז כלים סטטיסטי | `wilson_interval`, `pass_at_k`, `fisher_p_value` | 10 דק' |
| ב' | בדיקה לפי מחלקות שקילות | `failing_inputs` | 5 דק' |
| ג' | הניסוי: $x_A$ מול $x_B$ | — (הרצות בכלי AI) | 20 דק' |
| ד' | ניתוח | — | 8 דק' |
| ה' | מסקנה | — | 2 דק' |
| שיעורי בית | כמה זה יעלה? מבחן פישר ביד | `geometric`, `expected_cost`, `chain_costs`; חישוב ידני | בבית |

## איך עובדים

1. **מממשים** כל פונקציה שמסומנת `# TODO` — **בעצמכם, בלי כלי AI**. (בחלק ג' משתמשים בכלי AI, כי הוא נושא הניסוי.)
2. **מריצים את תא הבדיקה** שאחרי כל פונקציה. הבדיקות משתמשות **במספרים מההרצאה** — כשהמימוש נכון יודפס `✓`; אחרת תופיע הודעה שמסבירה מה נכשל.
3. **לפני כל מדידה כותבים תחזית** בתא המיועד לכך. ההבדל בין התחזית לתוצאה הוא חלק מהציון.
4. **עונים על שאלות הניתוח** במונחי ההרצאה ובסימונים שלה.

## סימונים (כמו בהרצאה)

| סימון | משמעות |
|---|---|
| $x_A,\ x_B$ | שני הפרומפטים שמשווים |
| $P_\text{succ}(x)$ | הסתברות ההצלחה (האמיתית, לא ידועה) של פרומפט $x$ |
| $R,\ c$ | מספר ההרצות של פרומפט; מספר ההרצות שהצליחו |
| $\hat{P}_\text{succ} = c / R$ | האומד של $P_\text{succ}$ מהנתונים |
| $z^\ast$ | ערך קריטי של ההתפלגות הנורמלית ($1.96$ לרמת סמך 95%) |
| $k$ | מספר הניסיונות שמותר לעשות (ב-pass@k) |
| $p_\text{val}$ | ערך p של מבחן פישר |
| $\Omega,\ \Omega_1, \dots, \Omega_6,\ \omega$ | תחום הקלט של `is_prime`; מחלקות השקילות שלו; קלט בודד |

## הגשה

בסוף: **Runtime → Restart session and run all**, ודאו שתא הבדיקה המסכם מראה `✓` לכל הפונקציות ושתאי הניסוי רצים, ואז **File → Download → Download .ipynb** והגישו.

</div>

In [1]:
# תא הכנה — הריצו פעם אחת בתחילת העבודה
import math
import signal
import time

print("✓ מוכן")

✓ מוכן


<div dir="rtl">

---

# חלק א' — ארגז כלים סטטיסטי (10 דק')

הרצתם פרומפט $R$ פעמים, ו-$c$ הרצות עברו את הבדיקות. שלוש שאלות, שלוש פונקציות:
1. מה אפשר לומר על $P_\text{succ}$? ← **רווח וילסון**.
2. מה הסיכוי להצליח אם מותר לנסות $k$ פעמים? ← **pass@k**.
3. האם פרומפט אחד באמת טוב מאחר? ← **מבחן פישר**.

## 1. רווח וילסון

**הנוסחה (מההרצאה, חלק י').** נכתוב $\hat{P} = c / R$:

$$\frac{\hat{P} + \dfrac{(z^\ast)^2}{2R} \;\pm\; z^\ast\sqrt{\dfrac{\hat{P}(1-\hat{P})}{R} + \dfrac{(z^\ast)^2}{4R^2}}}{1 + \dfrac{(z^\ast)^2}{R}}$$

עם $-$ מקבלים את הקצה התחתון, עם $+$ את העליון.

**דוגמה ידנית** — $c = 4$, $R = 5$ (השורה "4/5" בטבלה בהרצאה), $z^\ast = 1.96$, $(z^\ast)^2 = 3.8416$:
- $\hat{P} = 0.8$.
- מרכז (המונה בלי $\pm$): $0.8 + 3.8416 / 10 = 1.18416$.
- מכנה: $1 + 3.8416 / 5 = 1.76832$.
- חצי הרוחב: $1.96 \cdot \sqrt{0.8 \cdot 0.2 / 5 + 3.8416 / 100} = 1.96 \cdot \sqrt{0.070416} \approx 0.5201$.
- הרווח: $\left[\dfrac{1.18416 - 0.5201}{1.76832},\ \dfrac{1.18416 + 0.5201}{1.76832}\right] \approx [0.38,\ 0.96]$.

**שימו לב:** הקצוות חייבים להישאר בין 0 ל-1 (השתמשו ב-`max(0.0, ...)` וב-`min(1.0, ...)`).

</div>

In [2]:
def wilson_interval(c, R, z_star=1.96):
    """Wilson score interval for a success probability, from c successes out of R runs.
    returns: (low, high), both between 0 and 1.
    e.g. wilson_interval(4, 5) -> approximately (0.38, 0.96)
    """
    P_hat = c / R
    z2 = z_star ** 2
    centre = P_hat + z2 / (2 * R)                                     # numerator without the ±
    half = z_star * math.sqrt(P_hat * (1 - P_hat) / R + z2 / (4 * R ** 2))
    denom = 1 + z2 / R
    return max(0.0, (centre - half) / denom), min(1.0, (centre + half) / denom)

In [3]:
# בדיקה — רווח וילסון (הטבלה מההרצאה)
def check_wilson():
    def r2(t):
        return tuple(round(v, 2) for v in t)
    assert r2(wilson_interval(4, 5)) == (0.38, 0.96), "wilson_interval: הדוגמה הידנית 4/5"
    assert r2(wilson_interval(5, 5)) == (0.57, 1.00), "wilson_interval: 5/5 צריך לתת [0.57, 1.00] — לא [1, 1]"
    assert r2(wilson_interval(0, 5)) == (0.00, 0.43), "wilson_interval: 0/5 (קצה תחתון 0)"
    assert r2(wilson_interval(1, 5)) == (0.04, 0.62), "wilson_interval: 1/5"
    assert r2(wilson_interval(45, 50)) == (0.79, 0.96), "wilson_interval: 45/50 — רווח צר יותר כי R גדול"
    return True

check_wilson() and print("✓ wilson_interval")

✓ wilson_interval


<div dir="rtl">

## 2. pass@k

**הנוסחה (מההרצאה):**

$$\widehat{\text{pass@}k} = 1 - \frac{\binom{R-c}{k}}{\binom{R}{k}}$$

$\binom{R}{k}$ הוא מספר הדרכים לבחור $k$ הרצות מתוך $R$; $\binom{R-c}{k}$ — מספר הדרכים לבחור $k$ הרצות **רק מתוך הכישלונות**. היחס הוא ההסתברות ש-$k$ הרצות שנבחרו באקראי כולן נכשלו, והמשלים הוא pass@k.

**דוגמה ידנית:** $R = 5$, $c = 2$, $k = 3$: $\binom{3}{3} = 1$, $\binom{5}{3} = 10$, ולכן $\text{pass@}3 = 1 - 1/10 = 0.9$.

**כלי:** `math.comb(n, k)` מחשב $\binom{n}{k}$, ומחזיר 0 כש-$k > n$ — כלומר כשיש פחות כישלונות מ-$k$, הנוסחה נותנת אוטומטית pass@k = 1.

</div>

In [4]:
def pass_at_k(R, c, k):
    """Unbiased estimate of pass@k from R runs with c successes: 1 - C(R-c, k) / C(R, k)."""
    return 1 - math.comb(R - c, k) / math.comb(R, k)   # comb(n, k) = 0 when k > n -> pass@k = 1

In [5]:
# בדיקה — pass@k
def check_passk():
    assert abs(pass_at_k(5, 2, 3) - 0.9) < 1e-12, "pass_at_k: הדוגמה הידנית (R=5, c=2, k=3) צריכה לתת 0.9"
    assert abs(pass_at_k(5, 2, 1) - 0.4) < 1e-12, "pass_at_k: כש-k=1 התוצאה היא פשוט c/R"
    assert pass_at_k(5, 0, 3) == 0.0, "pass_at_k: בלי אף הצלחה — 0"
    assert pass_at_k(5, 4, 3) == 1.0, "pass_at_k: כשיש פחות כישלונות מ-k — 1"
    assert round(pass_at_k(10, 3, 5), 3) == 0.917, "pass_at_k: R=10, c=3, k=5"
    return True

check_passk() and print("✓ pass_at_k")

✓ pass_at_k


<div dir="rtl">

## 3. מבחן פישר

**הרעיון (מההרצאה, חלק י').** $x_A$ הצליח $c_A$ פעמים מתוך $R_A$ הרצות, ו-$x_B$ הצליח $c_B$ פעמים מתוך $R_B$; סך הכול $c_A + c_B$ הצלחות מתוך $R_A + R_B$ הרצות. **השערת האפס:** $P_\text{succ}(x_A) = P_\text{succ}(x_B)$. מבחן פישר שואל: בהינתן שהיו $c_A + c_B$ הצלחות, האם **החלוקה** שלהן בין $x_A$ ל-$x_B$ מעידה על הבדל אמיתי? אם השערת האפס נכונה, לתווית של הרצה ($x_A$ או $x_B$) אין קשר להצלחה, ולכן כל בחירה של **אילו** $c_A + c_B$ מתוך $R_A + R_B$ ההרצות הצליחו סבירה באותה מידה.

- **$a$:** מספר ההצלחות שנופלות בקבוצה של $x_A$ בחלוקה אקראית כזו. בפועל נצפה $a = c_A$.
- **הטווח של $a$:** מ-$a_\text{min} = \max(0,\ c_A + c_B - R_B)$ (ב-$x_B$ אין מקום ליותר מ-$R_B$ הצלחות) עד $a_\text{max} = \min(R_A,\ c_A + c_B)$ (ב-$x_A$ אין יותר מ-$R_A$ הרצות).
- **ההסתברות של כל $a$** (מכנה: מספר הדרכים לבחור אילו $c_A + c_B$ מתוך $R_A + R_B$ ההרצות הצליחו; מונה: מספר הבחירות שבהן בדיוק $a$ מההצלחות ב-$x_A$ — בוחרים $a$ מתוך $R_A$ הרצות של $x_A$, ואת $c_A + c_B - a$ ההצלחות האחרות מתוך $R_B$ הרצות של $x_B$):

$$\Pr(a) = \frac{\binom{R_A}{a}\binom{R_B}{\,c_A + c_B - a\,}}{\binom{R_A + R_B}{\,c_A + c_B\,}}$$

- **ערך ה-p (דו-צדדי):** "קיצונית לפחות כמו מה שנצפה" = "סבירה לכל היותר כמוה". סוכמים את $\Pr(a)$ על כל $a$ בטווח שמקיים $\Pr(a) \le \Pr(c_A)$ — משני הצדדים, כי לפני הניסוי לא ידוע איזה פרומפט טוב יותר:

$$p_\text{val} = \sum_{a:\ \Pr(a) \le \Pr(c_A)} \Pr(a)$$

בהרצאה ראיתם את החישוב המלא עבור $6/10$ מול $9/10$ ($p_\text{val} \approx 0.303$).

**דוגמה ידנית:** $c_A = 1, R_A = 5, c_B = 4, R_B = 5$. סך הכול 5 הצלחות מתוך 10 הרצות, $\binom{10}{5} = 252$:

| $a$ | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| $\binom{5}{a}\binom{5}{5-a}$ | 1 | 25 | 100 | 100 | 25 | 1 |
| $\Pr(a)$ | 0.004 | **0.099** | 0.397 | 0.397 | 0.099 | 0.004 |

נצפה $a = c_A = 1$, עם $\Pr(1) = 25/252 \approx 0.099$. הערכים שההסתברות שלהם $\le 0.099$ הם $a = 0, 1, 4, 5$, ולכן $p_\text{val} = (1 + 25 + 25 + 1) / 252 \approx 0.206$.

**טיפ נומרי:** כדי ששתי הסתברויות שוות לא "ייפלו" בגלל שגיאת עיגול, השוו `pr(a) <= pr(c_A) * (1 + 1e-9)`.

</div>

In [6]:
def fisher_p_value(c_A, R_A, c_B, R_B):
    """Two-sided Fisher exact test p-value for c_A/R_A vs c_B/R_B successes."""
    total = c_A + c_B                                    # all successes, in both groups

    def pr(a):                                           # Pr(a): a of the successes fall in group A
        return math.comb(R_A, a) * math.comb(R_B, total - a) / math.comb(R_A + R_B, total)

    observed = pr(c_A)
    a_min, a_max = max(0, total - R_B), min(R_A, total)
    return sum(pr(a) for a in range(a_min, a_max + 1) if pr(a) <= observed * (1 + 1e-9))

In [7]:
# בדיקה — מבחן פישר
def check_fisher():
    assert round(fisher_p_value(1, 5, 4, 5), 3) == 0.206, "fisher_p_value: הדוגמה הידנית 1/5 מול 4/5"
    assert round(fisher_p_value(6, 10, 9, 10), 3) == 0.303, "fisher_p_value: 6/10 מול 9/10 — הדוגמה מההרצאה"
    assert round(fisher_p_value(1, 5, 5, 5), 3) == 0.048, "fisher_p_value: 1/5 מול 5/5"
    assert round(fisher_p_value(0, 5, 5, 5), 3) == 0.008, "fisher_p_value: 0/5 מול 5/5"
    assert round(fisher_p_value(30, 50, 45, 50), 3) == 0.001, "fisher_p_value: 30/50 מול 45/50"
    assert abs(fisher_p_value(3, 5, 3, 5) - 1.0) < 1e-9, "fisher_p_value: תוצאות זהות — 1"
    return True

check_fisher() and print("✓ fisher_p_value")

✓ fisher_p_value


<div dir="rtl">

## הכלי המשולב (נתון)

`analyze` משתמשת בשלוש הפונקציות שלכם ומדפיסה סיכום להשוואה בין שני פרומפטים. הריצו אותה על הדוגמה הידנית ($1/5$ מול $4/5$) ובדקו שהמספרים מתאימים: רווחי וילסון כמו בטבלה בהרצאה, ו-$p_\text{val}$ כמו בדוגמה הידנית של מבחן פישר.

</div>

In [8]:
def analyze(c_A, R_A, c_B, R_B):
    """Prints Wilson interval, pass@3 and E[G] for each prompt, and the Fisher p-value."""
    for name, c, R in (("A", c_A, R_A), ("B", c_B, R_B)):
        low, high = wilson_interval(c, R)
        exp_runs = f"{R / c:.2f}" if c else "∞ (לא נמדדה הצלחה)"
        print(f"פרומפט {name}: c/R = {c}/{R} = {c / R:.2f} | רווח וילסון 95%: [{low:.2f}, {high:.2f}] | "
              f"pass@3 = {pass_at_k(R, c, 3):.2f} | E[G] ≈ R/c = {exp_runs}")
    print(f"מבחן פישר (דו-צדדי): p_val = {fisher_p_value(c_A, R_A, c_B, R_B):.4f}")


analyze(1, 5, 4, 5)

פרומפט A: c/R = 1/5 = 0.20 | רווח וילסון 95%: [0.04, 0.62] | pass@3 = 0.60 | E[G] ≈ R/c = 5.00
פרומפט B: c/R = 4/5 = 0.80 | רווח וילסון 95%: [0.38, 0.96] | pass@3 = 1.00 | E[G] ≈ R/c = 1.25
מבחן פישר (דו-צדדי): p_val = 0.2063


<div dir="rtl">

---

# חלק ב' — בדיקה לפי מחלקות שקילות (5 דק')

בהרצאה (חלק ג'3) חילקנו את תחום הקלט של `is_prime`, $\Omega = \mathbb{Z}$, למחלקות שקילות, ובחרנו נציגים — כולל **ערכי גבול**. הטבלה הזו היא גם המפרט וגם הבדיקה:

| מחלקה | תיאור | נציגים | פלט נדרש |
|---|---|---|---|
| $\Omega_1$ | $\omega \le 1$ | $-7, -1, 0, 1$ | `False` |
| $\Omega_2$ | $\omega = 2$ | $2$ | `True` |
| $\Omega_3$ | זוגיים, $\omega > 2$ | $4, 100$ | `False` |
| $\Omega_4$ | אי-זוגיים פריקים | $9, 25, 91, 561$ | `False` |
| $\Omega_5$ | ראשוניים אי-זוגיים עד מיליון | $3, 97, 7919$ | `True` |
| $\Omega_6$ | ראשוניים מעל מיליון (כולל דרישת ביצועים) | $1{,}000{,}000{,}007$ | `True` תוך פחות משנייה |

## המשימה

ממשו `failing_inputs(f, cases)`: מקבלת פונקציה `f` ומילון `cases` של $\omega \mapsto$ הפלט הנדרש, ומחזירה **רשימה של הקלטים שבהם `f` נכשלה**, לפי הסדר:
- אם `f(ω)` מחזירה ערך שונה מהנדרש — מוסיפים את $\omega$ עצמו.
- אם `f(ω)` **זורקת חריגה** — מוסיפים מחרוזת בצורה `"ω (שם החריגה)"`, למשל `"-7 (ValueError)"`. (שם החריגה: `type(e).__name__`.)

**למה חשוב לתפוס חריגות?** מימוש שמחשב `math.isqrt(n)` בלי לטפל קודם ב-$\omega \le 1$ זורק `ValueError` על מספר שלילי. בלי `try/except`, הבדיקה כולה הייתה קורסת במקום לדווח "נכשל ב-$\Omega_1$".

</div>

In [9]:
def failing_inputs(f, cases):
    """cases: dict omega -> expected output. Returns the list of inputs on which f fails, in order:
    omega itself if f(omega) != expected, or the string f"{omega} ({ExceptionName})" if f raised.
    """
    bad = []
    for omega, expected in cases.items():
        try:
            if f(omega) != expected:
                bad.append(omega)
        except Exception as e:
            bad.append(f"{omega} ({type(e).__name__})")
    return bad

In [10]:
# שלושה מימושים לדוגמה (מההרצאה) — לבדיקת הפונקציה שלכם
def naive_is_prime(n):                      # הפרומפט החלש, חלק א' בהרצאה
    for i in range(2, n):
        if n % i == 0:
            return False
    return True

def no_guard_is_prime(n):                   # עד שורש n, אבל בלי טיפול ב-n <= 1
    for d in range(2, math.isqrt(n) + 1):
        if n % d == 0:
            return False
    return True

def off_by_one_is_prime(n):                 # באג ה-off-by-one מחלק ג'3 בהרצאה
    if n <= 1:
        return False
    if n <= 3:
        return True
    if n % 2 == 0:
        return False
    for d in range(3, math.isqrt(n), 2):    # חסר + 1
        if n % d == 0:
            return False
    return True

def check_failing():
    omega1 = {-7: False, -1: False, 0: False, 1: False}
    omega4 = {9: False, 25: False, 91: False, 561: False}
    assert failing_inputs(naive_is_prime, omega1) == [-7, -1, 0, 1], "failing_inputs: naive נכשל בכל Ω1"
    assert failing_inputs(no_guard_is_prime, omega1) == ["-7 (ValueError)", "-1 (ValueError)", 0, 1], \
        "failing_inputs: חריגה צריכה להירשם כ-'ω (ValueError)'"
    assert failing_inputs(off_by_one_is_prime, omega4) == [9, 25], \
        "failing_inputs: off-by-one נכשל רק על הגבול (9, 25), לא על 91 ו-561"
    assert failing_inputs(off_by_one_is_prime, omega1) == [], "failing_inputs: בלי כישלונות — רשימה ריקה"
    return True

check_failing() and print("✓ failing_inputs")

✓ failing_inputs


<div dir="rtl">

## כלי הבדיקה המלא (נתון)

`evaluate(f)` מריצה את `failing_inputs` שלכם על מחלקות $\Omega_1$–$\Omega_5$, ובודקת את $\Omega_6$ עם הגבלת זמן של שנייה אחת. `record` מריצה את `evaluate` ושומרת את התוצאה ביומן הניסוי `LOG`.

**תחזית:** על אילו מחלקות ייכשל כל אחד משלושת המימושים לדוגמה?

**התחזית שלי (פתרון):** `naive_is_prime` — $\Omega_1$ (מחזיר `True` על $\omega \le 1$) ו-$\Omega_6$ (לולאה עד $\omega$, איטי מדי). `no_guard_is_prime` — רק $\Omega_1$, ועם `ValueError` על השליליים (בזכות השורש הוא מהיר, ולכן עובר את $\Omega_6$). `off_by_one_is_prime` — רק $\Omega_4$, על 9 ו-25 (ערכי הגבול).

</div>

In [11]:
CLASSES = {
    "Ω1: ω ≤ 1": {-7: False, -1: False, 0: False, 1: False},
    "Ω2: ω = 2": {2: True},
    "Ω3: זוגיים > 2": {4: False, 100: False},
    "Ω4: אי-זוגיים פריקים": {9: False, 25: False, 91: False, 561: False},
    "Ω5: ראשוניים אי-זוגיים ≤ 10⁶": {3: True, 97: True, 7919: True},
}
LARGE_PRIME = 1_000_000_007


def _timeout(signum, frame):
    raise TimeoutError


def evaluate(f, verbose=True):
    """Runs f on every equivalence class. Returns the list of failed class names (empty = PASS)."""
    failed = []
    for name, cases in CLASSES.items():
        bad = failing_inputs(f, cases)
        if bad:
            failed.append(name)
        if verbose:
            print(("✓ " if not bad else "✗ ") + name + ("" if not bad else f"  ← נכשל על: {bad}"))
    signal.signal(signal.SIGALRM, _timeout)
    signal.alarm(1)
    try:
        ok = f(LARGE_PRIME) is True
    except TimeoutError:
        ok = False
    except Exception:
        ok = False
    finally:
        signal.alarm(0)
    if not ok:
        failed.append("Ω6: ראשוני גדול")
    if verbose:
        print(("✓ " if ok else "✗ ") + "Ω6: ראשוני גדול (1,000,000,007) תוך פחות משנייה")
        print("תוצאה:", "PASS" if not failed else "FAIL")
    return failed


LOG = []   # יומן הניסוי: רשומה אחת לכל הרצה


def record(prompt, f, adapted=False):
    """Evaluates f and appends the result to LOG. prompt: "A" or "B"."""
    print(f"--- הרצה {len(LOG) + 1}, פרומפט {prompt}{' (נדרשה התאמת פורמט)' if adapted else ''} ---")
    failed = evaluate(f)
    LOG.append({"prompt": prompt, "pass": not failed, "failed": failed, "adapted": adapted})


for impl in (naive_is_prime, no_guard_is_prime, off_by_one_is_prime):
    print(f"=== {impl.__name__} ===")
    evaluate(impl)
    print()

=== naive_is_prime ===
✗ Ω1: ω ≤ 1  ← נכשל על: [-7, -1, 0, 1]
✓ Ω2: ω = 2
✓ Ω3: זוגיים > 2
✓ Ω4: אי-זוגיים פריקים
✓ Ω5: ראשוניים אי-זוגיים ≤ 10⁶


✗ Ω6: ראשוני גדול (1,000,000,007) תוך פחות משנייה
תוצאה: FAIL

=== no_guard_is_prime ===
✗ Ω1: ω ≤ 1  ← נכשל על: ['-7 (ValueError)', '-1 (ValueError)', 0, 1]
✓ Ω2: ω = 2
✓ Ω3: זוגיים > 2
✓ Ω4: אי-זוגיים פריקים
✓ Ω5: ראשוניים אי-זוגיים ≤ 10⁶
✓ Ω6: ראשוני גדול (1,000,000,007) תוך פחות משנייה
תוצאה: FAIL

=== off_by_one_is_prime ===
✓ Ω1: ω ≤ 1
✓ Ω2: ω = 2
✓ Ω3: זוגיים > 2
✗ Ω4: אי-זוגיים פריקים  ← נכשל על: [9, 25]
✓ Ω5: ראשוניים אי-זוגיים ≤ 10⁶
✓ Ω6: ראשוני גדול (1,000,000,007) תוך פחות משנייה
תוצאה: FAIL



<div dir="rtl">

---

# חלק ג' — הניסוי: $x_A$ מול $x_B$ (20 דק')

## פרוטוקול (קראו לפני שמתחילים)

1. **כל הרצה בשיחה חדשה** — אחרת ההרצות לא בלתי-תלויות, וכל הנוסחאות מחלק א' לא תקפות.
2. **אותו כלי AI לכל 10 ההרצות**, ו-$x_A$ ו-$x_B$ **לסירוגין** (A, B, A, B, ...), כדי ששינוי בכלי באמצע לא יפגע רק בפרומפט אחד.
3. **מעתיקים את הקוד כמו שהוא.** ההתאמות היחידות שמותרות: לשנות את שם הפונקציה ל-`is_prime`, ולמחוק שורות מחוץ לפונקציה (`input`, `print`, דוגמאות). הרצה שדרשה התאמה — מסמנים `adapted=True`.
4. **לא משנים את קוד הבדיקה ולא מדביקים אותו לפרומפט.**
5. **מחליטים מראש:** $R_A = R_B = 5$. לא מוסיפים פרומפטים באמצע.

## השערה ותחזית

**השערה** (במשפט אחד, בסימוני ההרצאה):

**ההשערה שלי (פתרון):** $P_\text{succ}(x_B) > P_\text{succ}(x_A)$ — לפרומפט המובנה יש הסתברות הצלחה גבוהה יותר.

**תחזית:** כמה מתוך 5 הרצות יעברו לכל פרומפט, כלומר מה $c_A$ ומה $c_B$?

**התחזית שלי (פתרון):** $c_A \approx 1$ (לפי מודל המכפלה מההרצאה, $P_\text{succ}(x_A) \approx 0.13$), $c_B \approx 4$ ($P_\text{succ}(x_B) \approx 0.815$, כלומר בערך 4 מתוך 5).

## הפרומפטים

**$x_A$** (מילה במילה): `תעשה בדיקת ראשוניות`

**$x_B$** — כתבו אותו לפי התבנית מההרצאה (חלק ד'). **לפני שכותבים**, מלאו את טבלת המיפוי: לכל מחלקה או החלטה — המשפט ב-$x_B$ שסוגר אותה.

| מחלקה / החלטה | המשפט ב-$x_B$ שסוגר אותה | אבן היסוד |
|---|---|---|
| $\Omega_1$: $\omega \le 1$ | | |
| $\Omega_2$–$\Omega_5$: נכונות | | |
| $\Omega_6$: ביצועים | | |
| פורמט: פונקציה טהורה בשם `is_prime` | | |
| שפה | | |

**$x_B$ שלי (פתרון — דוגמה):**

> כתוב פונקציה ב-Python בשם `is_prime(n)` שמקבלת מספר שלם ומחזירה `True` אם הוא ראשוני ו-`False` אחרת.
> - מספרים קטנים או שווים ל-1 (כולל 0 ושליליים) אינם ראשוניים. ($\Omega_1$)
> - 2 ראשוני; זוגיים גדולים מ-2 אינם ראשוניים. ($\Omega_2$, $\Omega_3$)
> - הפונקציה צריכה להיות יעילה גם למספרים בסדר גודל של מיליארד: בדיקה עד שורש n, כולל השורש עצמו (כך שגם 9 ו-25 ייבדקו נכון). ($\Omega_4$–$\Omega_6$)
> - רק ספרייה סטנדרטית.
> החזר: רק את הפונקציה, בלי `input()` ובלי `print` (פונקציה טהורה).

**טבלת המיפוי (פתרון):** $\Omega_1$ ← משפט 1 (אילוץ); $\Omega_2$–$\Omega_5$ ← משפטים 2–3 (אילוץ + מטרה); $\Omega_6$ ← משפט 3 (אילוץ ביצועים); פורמט ← השורה האחרונה (פורמט); שפה ← "ב-Python" (הקשר).

## עשר ההרצות

לכל הרצה: שיחה חדשה ← שליחת הפרומפט ← הדבקת הקוד שהתקבל **בתא של ההרצה** (במקום ההערה) ← הרצת התא. השורה האחרונה בכל תא, `record(...)`, בודקת את הקוד ורושמת את התוצאה ב-`LOG`. אם נדרשה התאמת פורמט — שנו ל-`adapted=True`.

</div>

In [12]:
# הרצה 1 — פרומפט A  (בפתרון: קוד מייצג להמחשה, לא פלט אמיתי של כלי)
def is_prime(n):
    for i in range(2, n):
        if n % i == 0:
            return False
    return True

record("A", is_prime, adapted=False)

--- הרצה 1, פרומפט A ---
✗ Ω1: ω ≤ 1  ← נכשל על: [-7, -1, 0, 1]
✓ Ω2: ω = 2
✓ Ω3: זוגיים > 2
✓ Ω4: אי-זוגיים פריקים
✓ Ω5: ראשוניים אי-זוגיים ≤ 10⁶


✗ Ω6: ראשוני גדול (1,000,000,007) תוך פחות משנייה
תוצאה: FAIL


In [13]:
# הרצה 2 — פרומפט B  (בפתרון: קוד מייצג להמחשה, לא פלט אמיתי של כלי)
import math

def is_prime(n: int) -> bool:
    """Return True if n is a prime number, False otherwise."""
    if n <= 1:
        return False
    if n <= 3:
        return True
    if n % 2 == 0:
        return False
    for d in range(3, math.isqrt(n) + 1, 2):
        if n % d == 0:
            return False
    return True

record("B", is_prime, adapted=False)

--- הרצה 2, פרומפט B ---
✓ Ω1: ω ≤ 1
✓ Ω2: ω = 2
✓ Ω3: זוגיים > 2
✓ Ω4: אי-זוגיים פריקים
✓ Ω5: ראשוניים אי-זוגיים ≤ 10⁶
✓ Ω6: ראשוני גדול (1,000,000,007) תוך פחות משנייה
תוצאה: PASS


In [14]:
# הרצה 3 — פרומפט A  (בפתרון: קוד מייצג להמחשה, לא פלט אמיתי של כלי)
def is_prime(n):
    for i in range(2, n):
        if n % i == 0:
            return False
    return True

# (נמחקה השורה n = int(input(...)) — התאמת פורמט)

record("A", is_prime, adapted=True)

--- הרצה 3, פרומפט A (נדרשה התאמת פורמט) ---
✗ Ω1: ω ≤ 1  ← נכשל על: [-7, -1, 0, 1]
✓ Ω2: ω = 2
✓ Ω3: זוגיים > 2
✓ Ω4: אי-זוגיים פריקים
✓ Ω5: ראשוניים אי-זוגיים ≤ 10⁶


✗ Ω6: ראשוני גדול (1,000,000,007) תוך פחות משנייה
תוצאה: FAIL


In [15]:
# הרצה 4 — פרומפט B  (בפתרון: קוד מייצג להמחשה, לא פלט אמיתי של כלי)
import math

def is_prime(n: int) -> bool:
    """Return True if n is a prime number, False otherwise."""
    if n <= 1:
        return False
    if n <= 3:
        return True
    if n % 2 == 0:
        return False
    for d in range(3, math.isqrt(n) + 1, 2):
        if n % d == 0:
            return False
    return True

record("B", is_prime, adapted=False)

--- הרצה 4, פרומפט B ---
✓ Ω1: ω ≤ 1
✓ Ω2: ω = 2
✓ Ω3: זוגיים > 2
✓ Ω4: אי-זוגיים פריקים
✓ Ω5: ראשוניים אי-זוגיים ≤ 10⁶
✓ Ω6: ראשוני גדול (1,000,000,007) תוך פחות משנייה
תוצאה: PASS


In [16]:
# הרצה 5 — פרומפט A  (בפתרון: קוד מייצג להמחשה, לא פלט אמיתי של כלי)
import math

def is_prime(n):
    for d in range(2, math.isqrt(n) + 1):
        if n % d == 0:
            return False
    return True

record("A", is_prime, adapted=False)

--- הרצה 5, פרומפט A ---
✗ Ω1: ω ≤ 1  ← נכשל על: ['-7 (ValueError)', '-1 (ValueError)', 0, 1]
✓ Ω2: ω = 2
✓ Ω3: זוגיים > 2
✓ Ω4: אי-זוגיים פריקים
✓ Ω5: ראשוניים אי-זוגיים ≤ 10⁶
✓ Ω6: ראשוני גדול (1,000,000,007) תוך פחות משנייה
תוצאה: FAIL


In [17]:
# הרצה 6 — פרומפט B  (בפתרון: קוד מייצג להמחשה, לא פלט אמיתי של כלי)
import math

def is_prime(n: int) -> bool:
    if n <= 1:
        return False
    if n <= 3:
        return True
    if n % 2 == 0:
        return False
    for d in range(3, math.isqrt(n), 2):
        if n % d == 0:
            return False
    return True

record("B", is_prime, adapted=False)

--- הרצה 6, פרומפט B ---
✓ Ω1: ω ≤ 1
✓ Ω2: ω = 2
✓ Ω3: זוגיים > 2
✗ Ω4: אי-זוגיים פריקים  ← נכשל על: [9, 25]
✓ Ω5: ראשוניים אי-זוגיים ≤ 10⁶
✓ Ω6: ראשוני גדול (1,000,000,007) תוך פחות משנייה
תוצאה: FAIL


In [18]:
# הרצה 7 — פרומפט A  (בפתרון: קוד מייצג להמחשה, לא פלט אמיתי של כלי)
def is_prime(n):
    for i in range(2, n):
        if n % i == 0:
            return False
    return True

record("A", is_prime, adapted=False)

--- הרצה 7, פרומפט A ---
✗ Ω1: ω ≤ 1  ← נכשל על: [-7, -1, 0, 1]
✓ Ω2: ω = 2
✓ Ω3: זוגיים > 2
✓ Ω4: אי-זוגיים פריקים
✓ Ω5: ראשוניים אי-זוגיים ≤ 10⁶


✗ Ω6: ראשוני גדול (1,000,000,007) תוך פחות משנייה
תוצאה: FAIL


In [19]:
# הרצה 8 — פרומפט B  (בפתרון: קוד מייצג להמחשה, לא פלט אמיתי של כלי)
import math

def is_prime(n: int) -> bool:
    """Return True if n is a prime number, False otherwise."""
    if n <= 1:
        return False
    if n <= 3:
        return True
    if n % 2 == 0:
        return False
    for d in range(3, math.isqrt(n) + 1, 2):
        if n % d == 0:
            return False
    return True

record("B", is_prime, adapted=False)

--- הרצה 8, פרומפט B ---
✓ Ω1: ω ≤ 1
✓ Ω2: ω = 2
✓ Ω3: זוגיים > 2
✓ Ω4: אי-זוגיים פריקים
✓ Ω5: ראשוניים אי-זוגיים ≤ 10⁶
✓ Ω6: ראשוני גדול (1,000,000,007) תוך פחות משנייה
תוצאה: PASS


In [20]:
# הרצה 9 — פרומפט A  (בפתרון: קוד מייצג להמחשה, לא פלט אמיתי של כלי)
def is_prime(n):
    if n < 2:
        return False
    d = 2
    while d * d <= n:
        if n % d == 0:
            return False
        d += 1
    return True

record("A", is_prime, adapted=False)

--- הרצה 9, פרומפט A ---
✓ Ω1: ω ≤ 1
✓ Ω2: ω = 2
✓ Ω3: זוגיים > 2
✓ Ω4: אי-זוגיים פריקים
✓ Ω5: ראשוניים אי-זוגיים ≤ 10⁶
✓ Ω6: ראשוני גדול (1,000,000,007) תוך פחות משנייה
תוצאה: PASS


In [21]:
# הרצה 10 — פרומפט B  (בפתרון: קוד מייצג להמחשה, לא פלט אמיתי של כלי)
import math

def is_prime(n: int) -> bool:
    """Return True if n is a prime number, False otherwise."""
    if n <= 1:
        return False
    if n <= 3:
        return True
    if n % 2 == 0:
        return False
    for d in range(3, math.isqrt(n) + 1, 2):
        if n % d == 0:
            return False
    return True

record("B", is_prime, adapted=False)

--- הרצה 10, פרומפט B ---
✓ Ω1: ω ≤ 1
✓ Ω2: ω = 2
✓ Ω3: זוגיים > 2
✓ Ω4: אי-זוגיים פריקים
✓ Ω5: ראשוניים אי-זוגיים ≤ 10⁶
✓ Ω6: ראשוני גדול (1,000,000,007) תוך פחות משנייה
תוצאה: PASS


<div dir="rtl">

---

# חלק ד' — ניתוח (8 דק')

התא הבא מסכם את `LOG`: סופר $c_A$ ו-$c_B$, מריץ את `analyze`, ומדפיס **טבלת כישלונות לפי מחלקה** — באיזו מחלקה נכשלה כל הרצה.

</div>

In [22]:
def summarize(log):
    R = {p: sum(1 for e in log if e["prompt"] == p) for p in "AB"}
    c = {p: sum(1 for e in log if e["prompt"] == p and e["pass"]) for p in "AB"}
    print(f"R_A = {R['A']}, c_A = {c['A']} | R_B = {R['B']}, c_B = {c['B']}\n")
    if R["A"] and R["B"]:
        analyze(c["A"], R["A"], c["B"], R["B"])
    names = list(CLASSES) + ["Ω6: ראשוני גדול"]
    print(f"\n{'מחלקה':28} | כישלונות A | כישלונות B")
    for name in names:
        fa = sum(name in e["failed"] for e in log if e["prompt"] == "A")
        fb = sum(name in e["failed"] for e in log if e["prompt"] == "B")
        print(f"{name:28} | {fa:^10} | {fb:^10}")
    ad = {p: sum(e["adapted"] for e in log if e["prompt"] == p) for p in "AB"}
    print(f"{'התאמת פורמט נדרשה':28} | {ad['A']:^10} | {ad['B']:^10}")


summarize(LOG)

R_A = 5, c_A = 1 | R_B = 5, c_B = 4

פרומפט A: c/R = 1/5 = 0.20 | רווח וילסון 95%: [0.04, 0.62] | pass@3 = 0.60 | E[G] ≈ R/c = 5.00
פרומפט B: c/R = 4/5 = 0.80 | רווח וילסון 95%: [0.38, 0.96] | pass@3 = 1.00 | E[G] ≈ R/c = 1.25
מבחן פישר (דו-צדדי): p_val = 0.2063

מחלקה                        | כישלונות A | כישלונות B
Ω1: ω ≤ 1                    |     4      |     0     
Ω2: ω = 2                    |     0      |     0     
Ω3: זוגיים > 2               |     0      |     0     
Ω4: אי-זוגיים פריקים         |     0      |     1     
Ω5: ראשוניים אי-זוגיים ≤ 10⁶ |     0      |     0     
Ω6: ראשוני גדול              |     3      |     0     
התאמת פורמט נדרשה            |     1      |     0     


<div dir="rtl">

## שאלות ניתוח

**(א) מובהקות.** מה $p_\text{val}$? האם ההבדל בין $x_A$ ל-$x_B$ מובהק ברמה 0.05? כתבו **מה מותר להסיק ומה לא**, והתייחסו לרווחי הסמך.

**תשובה (על נתוני ההמחשה):** $c_A / R_A = 1/5$, $c_B / R_B = 4/5$, $p_\text{val} \approx 0.21 > 0.05$ — ההבדל **לא מובהק**. רווחי הסמך, $[0.04, 0.62]$ ו-$[0.38, 0.96]$, חופפים. מותר לומר: הנתונים **תומכים בכיוון** של ההשערה ($x_B$ הצליח פי 4). אסור לומר: "הוכחנו ש-$x_B$ טוב יותר" — וגם לא "אין הבדל". כדי להכריע צריך יותר הרצות.

**(ב) מנגנון.** באיזו מחלקה $x_A$ נכשל הכי הרבה? איזה משפט ב-$x_B$ סגר אותה (לפי טבלת המיפוי)? בטאו זאת במונחי מודל המכפלה מההרצאה, $P_\text{succ}(x) = \prod_\ell g_\ell$: איזה $g_\ell$ עלה, ובערך לכמה?

**תשובה:** $x_A$ נכשל הכי הרבה ב-$\Omega_1$ (4 מתוך 5 הרצות), ואחר כך ב-$\Omega_6$ (3). במודל המכפלה זו החלטה 2 מההרצאה — "טיפול בקלט $\omega \le 1$", עם $g_2 \approx 0.5$ בהמחשה; כאן נמדד בערך $1/5 = 0.2$ (הרצה אחת מתוך 5 טיפלה נכון). המשפט הראשון ב-$x_B$ ("מספרים קטנים או שווים ל-1... אינם ראשוניים") סגר אותה: ב-$x_B$ אין אף כישלון ב-$\Omega_1$, כלומר $g_2$ עלה לקרוב ל-1. הכישלון היחיד של $x_B$ היה ב-$\Omega_4$ (באג ה-off-by-one) — החלטה שהמשפט על "כולל השורש עצמו" נועד לסגור, ולא הצליח בהרצה אחת.

**(ג) עלות.** הציבו במודל העלות מההרצאה, $\mathbb{E}[C] = C_\text{write} + C_\text{check} / P_\text{succ}(x)$, את הזמנים שלכם ($C_\text{write}$ לכל פרומפט, $C_\text{check}$ — כמה דקות לקח לבדוק תשובה) ואת $\hat{P}_\text{succ}$ שמדדתם (חשבו בתא שלמטה). אם $\hat{P}_\text{succ} = 0$ — מה המשמעות?

**תשובה:** עם הזמנים שבתא: $\mathbb{E}[C_A] = 0.25 + 2 / 0.2 = 10.25$ דקות, $\mathbb{E}[C_B] = 3 + 2 / 0.8 = 5.5$ דקות — $x_B$ חוסך כמחצית מהזמן, למרות שכתיבתו ארוכה פי 12. אם $\hat{P}_\text{succ} = 0$, המודל נותן $\mathbb{E}[C] = \infty$: לא נמדדה אף הצלחה, ולכן אין דרך לדעת כמה ניסיונות יידרשו — מה שמצדיק לשנות את הפרומפט, לא לנסות שוב.

**(ד) תקפות.** ציינו לפחות **שני** איומים על תקפות **הניסוי שלכם** (גורמים שאינם הפרומפט ועלולים להשפיע על התוצאה), ואיך הייתם מצמצמים כל אחד בניסוי הבא.

**תשובה (דוגמאות):** (1) **מעט הרצות** — $R = 5$ נותן רווחי סמך רחבים; מצמצמים בהגדלת $R$ או באיחוד נתונים. (2) **שינוי בכלי** — אם הכלי עודכן באמצע, הוא היה פוגע בשני הפרומפטים בערך באותה מידה, כי הרצנו לסירוגין; עדיין, מתעדים את הכלי ואת התאריך. (3) **הכללה** — כלי אחד ומשימה אחת; המסקנה לא תקפה לכל כלי ולכל משימה. (4) **התאמת פורמט** — אם הרצה דרשה התאמה, יש החלטה ידנית בתוך המדידה; מדווחים אותן בנפרד.

</div>

In [23]:
# תא חישוב לשאלה (ג) — זמנים להמחשה
C_write = {"A": 0.25, "B": 3.0}      # דקות לכתוב כל פרומפט
C_check = 2.0                         # דקות לבדוק תשובה אחת (הדבקה + הרצת record)
P_hat = {"A": 1 / 5, "B": 4 / 5}
for p in "AB":
    print(f"E[C_{p}] = {C_write[p]} + {C_check} / {P_hat[p]:.2f} = {C_write[p] + C_check / P_hat[p]:.1f} דקות")

E[C_A] = 0.25 + 2.0 / 0.20 = 10.2 דקות
E[C_B] = 3.0 + 2.0 / 0.80 = 5.5 דקות


<div dir="rtl">

---

# חלק ה' — מסקנה (2 דק')

1. **מסקנה** בשלושה משפטים: מה ההשערה, מה נמדד, מה מותר להסיק (כולל ההסתייגויות).

   **פתרון:** השערנו ש-$P_\text{succ}(x_B) > P_\text{succ}(x_A)$. מדדנו $c_A / R_A = 1/5$ ו-$c_B / R_B = 4/5$, בהרצות בלתי-תלויות ובבדיקה אוטומטית לפי מחלקות שקילות. הכיוון תומך בהשערה, אבל $p_\text{val} \approx 0.21$ — ההבדל לא מובהק ב-5 הרצות, והמסקנה מוגבלת לכלי ולמשימה שנבדקו.

2. **כיול:** השוו את התחזית שלכם ל-$c_A$ ול-$c_B$ לתוצאה. בכמה טעיתם?

   **פתרון:** בנתוני ההמחשה התחזית ($c_A \approx 1$, $c_B \approx 4$) התאימה בדיוק — אצל סטודנטים צפויים פערים, וזה בסדר; מה שנבדק הוא ההסבר לפער.

3. **פרומפט $x_C$:** לפי טבלת הכישלונות של $x_B$ — איזה משפט אחד הייתם מוסיפים? לאיזו מחלקה הוא מכוון, ומה התחזית שלכם ל-$\hat{P}_\text{succ}(x_C)$?

   **פתרון:** הכישלון היחיד של $x_B$ היה ב-$\Omega_4$. מוסיפים דוגמאות few-shot מהגבול: `is_prime(9) → False`, `is_prime(25) → False`, `is_prime(7919) → True`. תחזית: $\hat{P}_\text{succ}(x_C) \approx 1$.

---

# בונוס

**1. הריצו את $x_C$** חמש פעמים (`record("C", ...)` — שנו את `summarize` כך שישווה בין B ל-C, או קראו ל-`analyze` ישירות). האם המשפט שהוספתם שיפר את המחלקה שאליה כוון?

**2. איחוד נתונים:** אחדו את ה-`LOG` שלכם עם של שני חברים (סה"כ $R = 15$ לכל פרומפט) והריצו שוב את `analyze`. מה קרה לרווחי הסמך ול-$p_\text{val}$? ואיזה איום תקפות **חדש** נוצר כשמאחדים נתונים מכלים שונים?

</div>

In [24]:
# בונוס 2 — דוגמה לאיחוד נתונים (מספרים להמחשה): R = 15 לכל פרומפט
analyze(4, 15, 12, 15)

פרומפט A: c/R = 4/15 = 0.27 | רווח וילסון 95%: [0.11, 0.52] | pass@3 = 0.64 | E[G] ≈ R/c = 3.75
פרומפט B: c/R = 12/15 = 0.80 | רווח וילסון 95%: [0.55, 0.93] | pass@3 = 1.00 | E[G] ≈ R/c = 1.25
מבחן פישר (דו-צדדי): p_val = 0.0092


<div dir="rtl">

---

# שיעורי בית — חלק 1: כמה זה יעלה?

בהרצאה (חלק ו', "כמה ניסיונות זה יעלה?", וחלק ח', "אמינות של שרשרת") בנינו שלושה מודלים פשוטים. כאן תממשו אותם ותשתמשו בהם.

**ההגדרות (מההרצאה):**

| סימון | הגדרה |
|---|---|
| $P$ | הסתברות ההצלחה של הרצה אחת, $P = P_\text{succ}(x)$. ההרצות בלתי-תלויות (כל אחת בשיחה חדשה) |
| $G$ | מספר ההרצות עד ההצלחה הראשונה, כולל ההרצה המוצלחת |
| $k$ | מספר הניסיונות שמותר לעשות |
| $C_\text{write}$ | זמן (בדקות) לכתוב את הפרומפט |
| $C_\text{check}$ | זמן לבדוק תשובה אחת (קריאה + הרצה) |
| $C$ | הזמן הכולל עד קוד תקין |
| $M$ | מספר הצעדים במשימה; הצעדים ממוספרים $s = 1, \dots, M$ |
| $P_s$ | ההסתברות שצעד $s$ ייכתב נכון (הצעדים בלתי-תלויים) |

**הנוסחאות:**

1. $G$ מתפלג גיאומטרית:

$$\Pr[G = k] = (1 - P)^{k-1} P, \qquad \mathbb{E}[G] = \frac{1}{P}, \qquad \Pr[G \le k] = 1 - (1 - P)^k$$

2. מודל העלות — זמן הכתיבה ועוד מספר ההרצות כפול זמן הבדיקה:

$$\mathbb{E}[C] = C_\text{write} + \mathbb{E}[G] \cdot C_\text{check} = C_\text{write} + \frac{C_\text{check}}{P}$$

3. אמינות של שרשרת. משימה של $M$ צעדים עובדת רק אם כל הצעדים עובדים. יחידת עבודה אחת = כתיבה ובדיקה של צעד אחד.

$$\Pr[\text{all steps correct in one attempt}] = \prod_{s=1}^{M} P_s, \qquad \mathbb{E}[\text{cost}_\text{single}] = \frac{M}{\prod_{s=1}^{M} P_s}, \qquad \mathbb{E}[\text{cost}_\text{split}] = \sum_{s=1}^{M} \frac{1}{P_s}$$

- **פרומפט יחיד:** אם התשובה נכשלה לא יודעים איזה צעד שבור, ולכן מבקשים מחדש את כל $M$ הצעדים. כל ניסיון עולה $M$ יחידות, ומספר הניסיונות גיאומטרי עם הסתברות הצלחה $\prod_s P_s$.
- **פירוק:** כל צעד נבדק לבד, ואם נכשל חוזרים רק עליו. צעד $s$ דורש בממוצע $1/P_s$ ניסיונות של יחידה אחת.

**דוגמאות ידניות (מההרצאה):**
- $P = 0.126$ (הפרומפט החלש): $\mathbb{E}[G] = 1/0.126 \approx 7.94$, ו-$\Pr[G \le 3] = 1 - 0.874^3 \approx 0.332$.
- $C_\text{write} = 0.5$, $C_\text{check} = 3$, $P = 0.126$: $\mathbb{E}[C] = 0.5 + 3/0.126 \approx 24.3$ דקות.
- $M = 5$, $P_s = 0.9$ לכל צעד: $\prod_s P_s = 0.9^5 \approx 0.590$, $\mathbb{E}[\text{cost}_\text{single}] = 5/0.590 \approx 8.47$, $\mathbb{E}[\text{cost}_\text{split}] = 5 \cdot (1/0.9) \approx 5.56$.

## המשימה

ממשו שלוש פונקציות:
1. `geometric(P, k)` — מחזירה זוג: $\big(\mathbb{E}[G],\ \Pr[G \le k]\big)$.
2. `expected_cost(C_write, C_check, P)` — מחזירה את $\mathbb{E}[C]$.
3. `chain_costs(P_steps)` — מקבלת **רשימה** `P_steps` של ההסתברויות $P_1, \dots, P_M$ (האורך שלה הוא $M$), ומחזירה שלישייה: $\big(\prod_s P_s,\ \mathbb{E}[\text{cost}_\text{single}],\ \mathbb{E}[\text{cost}_\text{split}]\big)$.

</div>

In [25]:
def geometric(P, k):
    """G ~ Geometric(P). returns: (E[G], Pr[G <= k])."""
    return 1 / P, 1 - (1 - P) ** k                   # Pr[G <= k] = 1 - Pr[all k runs failed]


def expected_cost(C_write, C_check, P):
    """Expected total time until correct code: C_write + C_check / P."""
    return C_write + geometric(P, 1)[0] * C_check    # C_write + E[G] * C_check


def chain_costs(P_steps):
    """P_steps: list of P_1, ..., P_M.
    returns: (Pr[all steps correct in one attempt], E[cost_single], E[cost_split]).
    """
    M = len(P_steps)
    pr_all = math.prod(P_steps)
    return pr_all, M / pr_all, sum(1 / p for p in P_steps)

In [26]:
# בדיקה — שיעורי בית (המספרים מההרצאה)
def check_hw():
    def r(t, n):
        return tuple(round(v, n) for v in t)
    assert r(geometric(0.126, 3), 3) == (7.937, 0.332), "geometric: הפרומפט החלש — E[G] ≈ 7.94, Pr[G<=3] ≈ 0.332"
    assert r(geometric(0.815, 3), 3) == (1.227, 0.994), "geometric: הפרומפט החזק — E[G] ≈ 1.23, Pr[G<=3] ≈ 0.994"
    assert geometric(1.0, 1) == (1.0, 1.0), "geometric: P = 1 — הצלחה בהרצה הראשונה"
    assert round(expected_cost(0.5, 3, 0.126), 1) == 24.3, "expected_cost: הפרומפט החלש — 24.3 דקות"
    assert round(expected_cost(3, 3, 0.815), 1) == 6.7, "expected_cost: הפרומפט החזק — 6.7 דקות"
    assert r(chain_costs([0.95] * 5), 2) == (0.77, 6.46, 5.26), "chain_costs: P_s = 0.95, M = 5"
    assert r(chain_costs([0.9] * 5), 2) == (0.59, 8.47, 5.56), "chain_costs: P_s = 0.90, M = 5"
    assert r(chain_costs([0.8] * 5), 2) == (0.33, 15.26, 6.25), "chain_costs: P_s = 0.80, M = 5"
    assert r(chain_costs([1.0, 0.5]), 9) == (0.5, 4.0, 3.0), "chain_costs: צעדים עם P_s שונים — [1.0, 0.5]"
    single = chain_costs([0.9])
    assert abs(single[1] - single[2]) < 1e-12, "chain_costs: כש-M = 1 אין הבדל בין שתי הגישות"
    return True

check_hw() and print("✓ geometric, expected_cost, chain_costs")

✓ geometric, expected_cost, chain_costs


<div dir="rtl">

## הטבלאות מההרצאה (נתון) — השוו

התא הבא משתמש בפונקציות שלכם ומדפיס את שתי הטבלאות מההרצאה. **הפלט צריך להיות זהה להן.**

</div>

In [27]:
print(f"{'prompt':8} | {'P_succ':>6} | {'E[G]':>5} | {'Pr[G<=3]':>8} | {'E[C] (min)':>10}")
for name, P, C_write in (("weak", 0.126, 0.5), ("strong", 0.815, 3)):
    EG, within3 = geometric(P, 3)
    print(f"{name:8} | {P:6.3f} | {EG:5.2f} | {within3:8.3f} | {expected_cost(C_write, 3, P):10.1f}")

print(f"\n{'P_s':>4} | {'P_s^5':>6} | {'E[cost_single]':>14} | {'E[cost_split]':>13} | {'ratio':>5}")
for P_s in (0.95, 0.90, 0.80):
    pr_all, single, split = chain_costs([P_s] * 5)
    print(f"{P_s:4.2f} | {pr_all:6.3f} | {single:14.2f} | {split:13.2f} | {single / split:5.1f}")

prompt   | P_succ |  E[G] | Pr[G<=3] | E[C] (min)
weak     |  0.126 |  7.94 |    0.332 |       24.3
strong   |  0.815 |  1.23 |    0.994 |        6.7

 P_s |  P_s^5 | E[cost_single] | E[cost_split] | ratio
0.95 |  0.774 |           6.46 |          5.26 |   1.2
0.90 |  0.590 |           8.47 |          5.56 |   1.5
0.80 |  0.328 |          15.26 |          6.25 |   2.4


<div dir="rtl">

## שאלות

**(א) כמה ניסיונות צריך?** מה ה-$k$ הקטן ביותר שעבורו $\Pr[G \le k] \ge 0.9$ — עבור הפרומפט החלש ($P = 0.126$) ועבור החזק ($P = 0.815$)? חלצו את $k$ מהנוסחה ביד (רמז: לוגריתם, וזכרו ש-$\ln(1-P) < 0$), ואמתו בלולאה בתא שלמטה.

**תשובה:** $\Pr[G \le k] \ge 0.9 \iff (1-P)^k \le 0.1 \iff k \ln(1-P) \le \ln 0.1$. מחלקים ב-$\ln(1-P) < 0$, ולכן כיוון אי-השוויון מתהפך: $k \ge \dfrac{\ln 0.1}{\ln(1-P)}$, ו-$k$ הוא השלם הקטן ביותר שמקיים זאת.
- חלש: $\dfrac{\ln 0.1}{\ln 0.874} = \dfrac{-2.3026}{-0.1347} \approx 17.10$, ולכן $k = 18$. בדיקה: $1 - 0.874^{17} \approx 0.899 < 0.9$, ו-$1 - 0.874^{18} \approx 0.911$.
- חזק: $\dfrac{\ln 0.1}{\ln 0.185} = \dfrac{-2.3026}{-1.6874} \approx 1.36$, ולכן $k = 2$. בדיקה: $1 - 0.185 = 0.815 < 0.9$, ו-$1 - 0.185^2 \approx 0.966$.

כדי להגיע לביטחון של 90%, הפרומפט החלש דורש פי 9 יותר ניסיונות.

**(ב) נקודת איזון.** עם הזמנים מההרצאה ($C_\text{check} = 3$; $C_\text{write} = 0.5$ לפרומפט החלש ו-$3$ לחזק; לחזק $P = 0.815$): מאיזו הסתברות הצלחה $P$ של הפרומפט החלש הוא היה **זול יותר בתוחלת** מהחזק? פתרו את אי-השוויון ביד, ואמתו עם `expected_cost`. מה המשמעות, בהשוואה ל-$P = 0.126$?

**תשובה:** $\mathbb{E}[C_\text{strong}] = 3 + 3/0.815 \approx 6.681$. הפרומפט החלש זול יותר בתוחלת כאשר $0.5 + 3/P < 6.681$, כלומר $3/P < 6.181$, כלומר $P > 3/6.181 \approx 0.485$. אימות: עם $P = 0.48$ מתקבל $\mathbb{E}[C_\text{weak}] = 6.75 > 6.681$, ועם $P = 0.49$ מתקבל $6.62 < 6.681$.

**משמעות:** החיסכון בכתיבה (2.5 דקות) משתלם רק אם הפרומפט החלש מצליח כמעט בכל הרצה שנייה — פי 3.9 מה-$0.126$ שלו. כל עוד $P_\text{succ}(x_\text{weak}) < 0.485$, כדאי להשקיע בכתיבת הפרומפט.

**(ג) הוכחה.** הוכיחו שלכל $M \ge 1$ ולכל $P_1, \dots, P_M \in (0, 1]$ מתקיים $\mathbb{E}[\text{cost}_\text{split}] \le \mathbb{E}[\text{cost}_\text{single}]$. מתי מתקיים שוויון? (רמז: לכל $s$, $\prod_{s'=1}^{M} P_{s'} \le P_s$.)

**תשובה:** יהי $s \in \{1, \dots, M\}$. כל $P_{s'} \le 1$, ולכן
$$\prod_{s'=1}^{M} P_{s'} = P_s \cdot \prod_{s' \ne s} P_{s'} \le P_s \quad\Longrightarrow\quad \frac{1}{P_s} \le \frac{1}{\prod_{s'=1}^{M} P_{s'}}$$
(כל הגדלים חיוביים, ולכן מותר להפוך). סוכמים את אי-השוויון על $s = 1, \dots, M$:
$$\mathbb{E}[\text{cost}_\text{split}] = \sum_{s=1}^{M} \frac{1}{P_s} \le \frac{M}{\prod_{s'=1}^{M} P_{s'}} = \mathbb{E}[\text{cost}_\text{single}] \qquad \blacksquare$$
**שוויון** מתקיים אם ורק אם יש שוויון לכל $s$, כלומר $\prod_{s' \ne s} P_{s'} = 1$ לכל $s$. זה קורה כש-$M = 1$ (המכפלה הריקה שווה 1), או כש-$M \ge 2$ וכל $P_s = 1$. **מסקנה:** במודל הזה פירוק אף פעם לא עולה יותר, והיתרון שלו גדל ככל ש-$P_s$ קטנים (עמודת היחס בטבלה: 1.2, 1.5, 2.4).

</div>

In [28]:
# (א) the smallest k with Pr[G <= k] >= 0.9
for name, P in (("weak", 0.126), ("strong", 0.815)):
    k = 1
    while geometric(P, k)[1] < 0.9:
        k += 1
    bound = math.log(0.1) / math.log(1 - P)
    print(f"{name}: k = {k} (ln(0.1) / ln(1 - P) = {bound:.2f}), Pr[G <= {k}] = {geometric(P, k)[1]:.3f}")

# (ב) break-even P of the weak prompt
E_strong = expected_cost(3, 3, 0.815)
P_even = 3 / (E_strong - 0.5)
print(f"\nE[C_strong] = {E_strong:.3f}; break-even P = 3 / ({E_strong:.3f} - 0.5) = {P_even:.3f}")
for P in (0.48, 0.49):
    print(f"P = {P}: E[C_weak] = {expected_cost(0.5, 3, P):.3f}")

weak: k = 18 (ln(0.1) / ln(1 - P) = 17.10), Pr[G <= 18] = 0.911
strong: k = 2 (ln(0.1) / ln(1 - P) = 1.36), Pr[G <= 2] = 0.966

E[C_strong] = 6.681; break-even P = 3 / (6.681 - 0.5) = 0.485
P = 0.48: E[C_weak] = 6.750
P = 0.49: E[C_weak] = 6.622


<div dir="rtl">

---

# שיעורי בית — חלק 2: מבחן פישר ביד

בהרצאה (חלק י') ראיתם את ההתפלגות המלאה של $a$ עבור $6/10$ מול $9/10$, ובחלק א' של המחברת — עבור $1/5$ מול $4/5$. כאן תחשבו ביד את המקרה $c_A = 1,\ R_A = 5,\ c_B = 5,\ R_B = 5$.

**(ד) חישוב ידני.**
1. כמה הצלחות היו בסך הכול, ומה המכנה $\binom{R_A + R_B}{c_A + c_B}$?
2. מה $a_\text{min}$ ו-$a_\text{max}$? הסבירו במשפט אחד למה $a = 0$ **אינו** אפשרי כאן.
3. מלאו טבלה: לכל $a$ בטווח — המונה $\binom{R_A}{a}\binom{R_B}{c_A + c_B - a}$ ו-$\Pr(a)$. ודאו שסכום המונים שווה למכנה.
4. אילו ערכי $a$ נכנסים לסכום של $p_\text{val}$? חשבו את $p_\text{val}$ כשבר ובעשרוני, ואמתו עם `fisher_p_value` בתא שלמטה.
5. האם ההבדל מובהק ברמה 0.05? הסבירו למה $a = 5$ נכלל בסכום, למרות שבו $x_A$ **מוצלח יותר** מ-$x_B$.

**תשובה:**
1. $c_A + c_B = 6$ הצלחות מתוך $R_A + R_B = 10$ הרצות; המכנה $\binom{10}{6} = 210$.
2. $a_\text{min} = \max(0,\ 6 - 5) = 1$, $a_\text{max} = \min(5,\ 6) = 5$. $a = 0$ אינו אפשרי: אז כל 6 ההצלחות היו ב-$x_B$, אבל ל-$x_B$ יש רק 5 הרצות.
3. הטבלה:

| $a$ | 1 (נצפה) | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| $\binom{5}{a}\binom{5}{6-a}$ | $5 \cdot 1 = 5$ | $10 \cdot 5 = 50$ | $10 \cdot 10 = 100$ | $5 \cdot 10 = 50$ | $1 \cdot 5 = 5$ |
| $\Pr(a)$ | 0.024 | 0.238 | 0.476 | 0.238 | 0.024 |

סכום המונים: $5 + 50 + 100 + 50 + 5 = 210$ — שווה למכנה.

4. $\Pr(c_A) = \Pr(1) = 5/210 \approx 0.024$. רק $a = 1$ ו-$a = 5$ מקיימים $\Pr(a) \le 5/210$, ולכן $p_\text{val} = (5 + 5)/210 = 1/21 \approx 0.048$; `fisher_p_value(1, 5, 5, 5)` מחזירה $0.0476$.
5. כן, $0.048 < 0.05$ — מובהק, בקושי. $a = 5$ נכלל כי המבחן דו-צדדי: לפני הניסוי לא ידענו איזה פרומפט טוב יותר, ולכן החלוקה 5 מול 1 לטובת $x_A$ קיצונית בדיוק כמו 1 מול 5 לטובת $x_B$ ($\Pr(5) = \Pr(1)$).

</div>

In [29]:
# תא לשאלה (ד)
c_A, R_A, c_B, R_B = 1, 5, 5, 5
total = c_A + c_B
den = math.comb(R_A + R_B, total)
nums = {a: math.comb(R_A, a) * math.comb(R_B, total - a) for a in range(max(0, total - R_B), min(R_A, total) + 1)}
for a, num in nums.items():
    print(f"a = {a}: numerator = {num:3d}, Pr(a) = {num / den:.4f}")
print("sum of numerators =", sum(nums.values()), "| denominator =", den)
print(f"p_val = {fisher_p_value(c_A, R_A, c_B, R_B):.4f}")

a = 1: numerator =   5, Pr(a) = 0.0238
a = 2: numerator =  50, Pr(a) = 0.2381
a = 3: numerator = 100, Pr(a) = 0.4762
a = 4: numerator =  50, Pr(a) = 0.2381
a = 5: numerator =   5, Pr(a) = 0.0238
sum of numerators = 210 | denominator = 210
p_val = 0.0476


<div dir="rtl">

---

# קריטריוני הצלחה ותוצר להגשה

- ✓ ארבע הפונקציות (`wilson_interval`, `pass_at_k`, `fisher_p_value`, `failing_inputs`) מומשו **בלי AI**, וכל תאי הבדיקה מדפיסים `✓`.
- ✓ 10 הרצות בשיחות נפרדות, לסירוגין, כולן רשומות ב-`LOG` — כולל `adapted=True` היכן שנדרשה התאמת פורמט.
- ✓ טבלת המיפוי בחלק ג' מראה משפט ב-$x_B$ לכל מחלקה.
- ✓ הפרשנות הסטטיסטית נכונה: אין "הוכחנו" כשההבדל לא מובהק, ואין "אין הבדל" כש-$p_\text{val} > 0.05$.
- ✓ שאלה (ב) בחלק ד' מקשרת בין מחלקה, משפט בפרומפט ו-$g_\ell$.
- ✓ לפחות שני איומים על תקפות, עם דרך לצמצם כל אחד.
- ✓ שיעורי בית: שלוש הפונקציות מומשו בלי AI, תא הבדיקה שלהן מדפיס `✓`, והטבלאות זהות להרצאה; לשאלות (א), (ב) ו-(ד) יש חישוב ידני ואימות בקוד, ולשאלה (ג) — הוכחה.
- ✓ לפחות הרצה אחת של $x_B$ עברה (PASS). אם אף אחת לא עברה — סבב תיקון אחד לפי חלק ו' בהרצאה (שגיאה/פלט + קלט + ציפייה), מתועד בתא נוסף.

**תוצר להגשה:** המחברת עצמה, אחרי **Restart session and run all** — עם המימושים, עשר ההרצות, פלט `summarize`, טבלת המיפוי, $x_B$ המלא ותשובות לכל השאלות.

</div>

<div dir="rtl">

---

# בדיקה מסכמת — הריצו לפני ההגשה

</div>

In [30]:
for name, check in [("wilson_interval", check_wilson),
                    ("pass_at_k", check_passk),
                    ("fisher_p_value", check_fisher),
                    ("failing_inputs", check_failing),
                    ("geometric, expected_cost, chain_costs", check_hw)]:
    try:
        check()
        print(f"✓ {name}")
    except NotImplementedError:
        print(f"✗ {name}  ← עדיין לא מומש")
    except AssertionError as e:
        print(f"✗ {name}  ← {e}")
print(f"\nהרצות ביומן הניסוי: {len(LOG)} (נדרשות 10)")

✓ wilson_interval
✓ pass_at_k
✓ fisher_p_value
✓ failing_inputs
✓ geometric, expected_cost, chain_costs

הרצות ביומן הניסוי: 10 (נדרשות 10)
